# BS5 curriculum pilot on Kaggle

Upload the NEW source ZIP as a Dataset; Kaggle extracts it automatically. Add Input, then run the four cells. Source stays in `/kaggle/input`; all writes go to `/kaggle/working`.

Cell 3 first performs preflight and an isolated 3,072-step debug matrix through every stage with full logging/shadows, plus checkpoint-free bundle acceptance. Only after these checks pass does it run the eight main runs: C-R/D-R/C-N/D-N x seeds 551/552 x 24,000 steps. No budget auto-reduction. Estimated runtime is informational, not a guarantee.

Curriculum budgets: S1 3k, S2 3k, S3 6k, S4 12k. Gates are diagnostic only. Replay clears on physical task changes; model/Adam/targets/global epsilon continue. S3d and S4 are physically identical, so their label change does not clear replay. This is a combined curriculum+replay-reset method.

Change RUN_NAME for a new independent experiment. Cell 3 resumes committed evaluation boundaries while the output folder survives. Across sessions, add the extracted repro_full Dataset and set RESTORE_FROM in cell 2 to its protocol.json parent; keep the SAME source version and device. audit_light contains no checkpoints and cannot resume. `/kaggle/working` may disappear between sessions, so download repro_full or save outputs before ending the session.

Enable GPU before cell 2 if desired. Kaggle NumPy/PyTorch are reused; no network installation is required. Run cell 4 after the training process stops to export complete or explicitly incomplete evidence.


## 1. Find the already extracted Dataset source


In [ ]:
from pathlib import Path
import os
import sys

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
INPUT = Path("/kaggle/input")
SOURCE_FILTER = ""  # Optional dataset name if multiple source versions exist.
candidates = sorted(
    {
        p.parents[2]
        for p in INPUT.rglob("run_bs5_curriculum_pilot.py")
        if p.parent.name == "training"
        and p.parents[1].name == "scripts"
        and SOURCE_FILTER in str(p)
        and all(
            (p.parents[2] / name).is_file()
            for name in (
                "src/envs/bs5_curriculum_env.py",
                "configs/bs5_curriculum_fixed_v1.json",
                "AGENTS.md",
            )
        )
    }
)
if len(candidates) != 1:
    raise RuntimeError(
        f"Expected one source root, found {len(candidates)}: {candidates}. "
        "Add the source Dataset or set SOURCE_FILTER."
    )
ROOT = candidates[0].resolve()
print("Source:", ROOT)

## 2. Change directory and choose writable output/device


In [ ]:
import json
import torch

os.chdir(ROOT)
WORK = Path("/kaggle/working")
RUN_NAME = "bs5_curriculum_fixed_20261023"  # Change for an independent experiment.
if not RUN_NAME or RUN_NAME in {".", ".."} or any(c in RUN_NAME for c in "/\\"):
    raise ValueError("RUN_NAME must be a single folder name")
OUT = WORK / RUN_NAME
CFG = WORK / f"{RUN_NAME}_config.json"
PROTOCOL = OUT / "protocol.json"
WORK.mkdir(parents=True, exist_ok=True)
# Optional: path to an already extracted repro_full Dataset from a prior session.
RESTORE_FROM = ""
if RESTORE_FROM and not OUT.exists():
    import shutil
    previous = Path(RESTORE_FROM)
    if not (previous / "protocol.json").is_file():
        raise RuntimeError("RESTORE_FROM must contain protocol.json")
    shutil.copytree(previous, OUT)
    source_zip = OUT / "source/uav_thesis_source.zip"
    if not source_zip.is_file():
        raise RuntimeError("Full resume needs source/uav_thesis_source.zip; audit_light cannot resume")
    shutil.copy2(source_zip, OUT / "source_snapshot.zip")
if PROTOCOL.is_file():
    config = json.loads(PROTOCOL.read_text(encoding="utf-8"))
    print("Reusing frozen protocol; device:", config["device"])
else:
    if OUT.exists():
        raise RuntimeError("Incomplete preparation: preserve this folder and choose a new RUN_NAME")
    config = json.loads((ROOT / "configs/bs5_curriculum_fixed_v1.json").read_text(encoding="utf-8"))
    config["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    CFG.write_text(json.dumps(config, indent=2), encoding="utf-8")
if config["device"].startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("This protocol needs CUDA. Enable a GPU accelerator before resuming.")
print("CWD:", Path.cwd())
print("Output:", OUT)
print("Device:", config["device"], "| torch:", torch.__version__)

## 3. Prepare once, then train or resume


In [ ]:
import subprocess

BASE = [sys.executable, "-B", "-u", "-m", "scripts.training.run_bs5_curriculum_pilot"]


def run_bs5(*args):
    subprocess.run(BASE + list(args), cwd=ROOT, check=True, env=os.environ.copy())


if not PROTOCOL.is_file():
    run_bs5("--prepare", "--config", str(CFG), "--outdir", str(OUT))
protocol = json.loads(PROTOCOL.read_text(encoding="utf-8"))
print("Frozen steps/run:", protocol["steps_per_run"], flush=True)
print("Smoke acceptance:", json.loads((OUT / "preflight/smoke_acceptance.json").read_text()), flush=True)
run_bs5("--run", "--resume", "--outdir", str(OUT))

## 4. Export ZIP bundles for download


In [ ]:
import shutil
import zipfile
from IPython.display import FileLink, display

# Run only after cell 3 has finished or stopped.
subprocess.run(
    [
        sys.executable,
        "-B",
        "-u",
        "-m",
        "scripts.training.run_bs5_curriculum_pilot",
        "--export",
        "--outdir",
        str(OUT),
    ],
    cwd=ROOT,
    check=True,
    env=os.environ.copy(),
)
registry = json.loads((OUT / "run_registry.json").read_text(encoding="utf-8"))
print("Experiment status:", registry["status"])
os.chdir(WORK)
for kind in ("audit_light", "repro_full"):
    source = OUT / "exports" / f"{kind}.zip"
    with zipfile.ZipFile(source) as bundle:
        bad = bundle.testzip()
        if bad is not None:
            raise RuntimeError(f"Corrupt ZIP member: {bad}")
    destination = WORK / f"{RUN_NAME}_{kind}.zip"
    shutil.copy2(source, destination)
    print(destination, f"({destination.stat().st_size / 1024**2:.1f} MiB)")
    display(FileLink(destination.name))